# Motion-S Kaggle Pipeline

This notebook generates 6 RVQ token layers for every test sample in the Motion-S competition.

Model roles:
- `rvq_vae_best.pth` is loaded only for token decoding / verification
- `length_estimator.pth` predicts the per-sample motion length from a CLIP text embedding
- `t5-small` is fine-tuned, or used as a frozen baseline, to map gloss text to flattened RVQ token strings

The notebook follows the competition flow end to end:
1. Load data
2. Load models
3. Predict motion length
4. Fine-tune T5-small if time allows
5. Generate 6 layers of RVQ tokens
6. Validate lengths and token ranges
7. Write `submission.csv`


## Requirements

Install the core runtime packages if they are not already available in the Kaggle environment:

- `torch`
- `transformers`
- `pandas`
- `numpy`
- `tqdm`
- `scipy`
- `sentencepiece`

Optional helpers for local experimentation:

- `accelerate`
- `huggingface-hub`
- `kagglehub`

The notebook expects the following model files to be mounted or discoverable locally:

- `rvq_vae_best.pth`
- `length_estimator.pth`
- `t5-small`


In [7]:
# Uncomment only if your Kaggle runtime is missing any of the required packages.
# %pip install -q torch transformers pandas numpy tqdm scipy sentencepiece accelerate

from IPython.display import display

In [8]:
import os
import random
import sys
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer, T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MIN_LENGTH = 40
MAX_LENGTH = 800
NUM_RVQ_LAYERS = 6
RVQ_VOCAB_SIZE = 512
LAYER_TOKENS = [f"<layer_{idx}>" for idx in range(NUM_RVQ_LAYERS)]
RVQ_TOKENS = [f"<rvq_{idx}>" for idx in range(RVQ_VOCAB_SIZE)]
TOKEN_COLUMNS = ["base_tokens", "residual_1", "residual_2", "residual_3", "residual_4", "residual_5"]


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



def ensure_dir(path: str | Path) -> Path:
    path_obj = Path(path)
    path_obj.mkdir(parents=True, exist_ok=True)
    return path_obj



def find_file_by_name(filename: str, roots: Sequence[str | Path]) -> Path | None:
    for root in roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        for candidate in root_path.rglob(filename):
            if candidate.is_file():
                return candidate
    return None



def resolve_model_file(filename: str, env_var: str | None = None, default_roots: Sequence[str | Path] = (Path("/kaggle/input"), Path.cwd())) -> Path:
    if env_var:
        configured = os.environ.get(env_var)
        if configured:
            configured_path = Path(configured)
            if configured_path.is_file():
                return configured_path
            if configured_path.is_dir():
                match = find_file_by_name(filename, [configured_path])
                if match is not None:
                    return match

    match = find_file_by_name(filename, default_roots)
    if match is not None:
        return match
    raise FileNotFoundError(f"Could not find {filename}. Set {env_var} or place the file under /kaggle/input.")



def build_gloss_prompt(gloss: str) -> str:
    return "" if pd.isna(gloss) else str(gloss).strip()



def normalize_layer_tokens(token_layers: Sequence[Sequence[int]]) -> list[list[int]]:
    lengths = [len(layer) for layer in token_layers]
    if not lengths:
        return [[] for _ in range(NUM_RVQ_LAYERS)]
    target_length = min(lengths)
    return [list(layer[:target_length]) for layer in token_layers]



def encode_rvq_layers(token_layers: Sequence[Sequence[int]]) -> str:
    layers = normalize_layer_tokens(token_layers)
    chunks: list[str] = []
    for layer_idx, layer in enumerate(layers):
        chunks.append(LAYER_TOKENS[layer_idx])
        chunks.extend(f"<rvq_{int(token)}>" for token in layer)
    return " ".join(chunks)



def clip_length(length: int, min_len: int = MIN_LENGTH, max_len: int = MAX_LENGTH) -> int:
    return int(max(min_len, min(max_len, int(length))))



def parse_token_string(token_string: object) -> list[int]:
    if token_string is None:
        return []
    if isinstance(token_string, float) and np.isnan(token_string):
        return []
    if isinstance(token_string, list):
        return [int(token) for token in token_string]
    text = str(token_string).strip()
    if not text:
        return []
    return [int(token) for token in text.split()]



def load_motion_dataframe(csv_path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    for column in ["sentence", "gloss"]:
        if column in df.columns:
            df[column] = df[column].fillna("")

    for column in TOKEN_COLUMNS:
        if column in df.columns:
            df[column] = df[column].apply(parse_token_string)

    if "length" not in df.columns:
        if "base_tokens" in df.columns:
            df["length"] = df["base_tokens"].apply(len)
        else:
            df["length"] = np.nan
    return df



def group_aware_train_val_split(
    df: pd.DataFrame,
    group_col: str = "signer_id",
    val_fraction: float = 0.1,
    seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if group_col not in df.columns:
        shuffled = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        split_idx = max(1, int(len(shuffled) * (1.0 - val_fraction)))
        return shuffled.iloc[:split_idx].reset_index(drop=True), shuffled.iloc[split_idx:].reset_index(drop=True)

    groups = pd.Index(df[group_col].fillna("__missing__").unique()).tolist()
    rng = np.random.default_rng(seed)
    rng.shuffle(groups)
    num_val_groups = max(1, int(round(len(groups) * val_fraction)))
    val_groups = set(groups[:num_val_groups])
    val_df = df[df[group_col].fillna("__missing__").isin(val_groups)].reset_index(drop=True)
    train_df = df[~df[group_col].fillna("__missing__").isin(val_groups)].reset_index(drop=True)
    return train_df, val_df



def token_histogram_features(token_bundle: torch.Tensor, mask: torch.Tensor | None = None) -> np.ndarray:
    bundle = torch.as_tensor(token_bundle, dtype=torch.long)
    if bundle.dim() == 2:
        bundle = bundle.unsqueeze(0)

    if mask is None:
        mask_tensor = torch.ones((bundle.size(0), bundle.size(-1)), dtype=torch.bool)
    else:
        mask_tensor = torch.as_tensor(mask, dtype=torch.bool)

    feature_rows: list[np.ndarray] = []
    for sample_idx in range(bundle.size(0)):
        sample = bundle[sample_idx]
        valid = mask_tensor[sample_idx]
        layer_features: list[np.ndarray] = []
        for layer_idx in range(sample.size(0)):
            layer_tokens = sample[layer_idx][valid]
            if layer_tokens.numel() == 0:
                hist = torch.zeros(RVQ_VOCAB_SIZE, dtype=torch.float32)
            else:
                hist = torch.bincount(layer_tokens.clamp(0, RVQ_VOCAB_SIZE - 1), minlength=RVQ_VOCAB_SIZE).float()
            hist = hist / hist.sum().clamp_min(1.0)
            layer_features.append(hist.numpy())
        feature_rows.append(np.concatenate(layer_features, axis=0))
    return np.stack(feature_rows, axis=0)



def approximate_fid(real_features: np.ndarray, generated_features: np.ndarray) -> float:
    real_features = np.atleast_2d(np.asarray(real_features, dtype=np.float64))
    generated_features = np.atleast_2d(np.asarray(generated_features, dtype=np.float64))
    mu_1 = real_features.mean(axis=0)
    mu_2 = generated_features.mean(axis=0)
    sigma_1 = np.cov(real_features, rowvar=False)
    sigma_2 = np.cov(generated_features, rowvar=False)
    eps = 1e-6
    try:
        from scipy import linalg as scipy_linalg

        eye = np.eye(sigma_1.shape[0], dtype=np.float64)
        covmean = scipy_linalg.sqrtm((sigma_1 + eps * eye) @ (sigma_2 + eps * eye))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        diff = mu_1 - mu_2
        return float(diff @ diff + np.trace(sigma_1 + sigma_2 - 2.0 * covmean))
    except Exception:
        diff = mu_1 - mu_2
        return float(diff @ diff + np.trace(sigma_1 + sigma_2))



def diversity_score(features: np.ndarray, sample_size: int = 1024) -> float:
    if len(features) < 2:
        return 0.0
    rng = np.random.default_rng(42)
    indices = rng.choice(len(features), size=min(sample_size, len(features)), replace=False)
    sample = np.asarray(features)[indices]
    distances = np.linalg.norm(sample[:, None, :] - sample[None, :, :], axis=-1)
    tri = distances[np.triu_indices_from(distances, k=1)]
    return float(tri.mean()) if len(tri) else 0.0



def r_precision_at_k(text_features: np.ndarray, motion_features: np.ndarray, k: int = 3, group_size: int = 32) -> float:
    text_features = np.atleast_2d(np.asarray(text_features, dtype=np.float64))
    motion_features = np.atleast_2d(np.asarray(motion_features, dtype=np.float64))
    num_samples = min(text_features.shape[0], motion_features.shape[0])
    text_features = text_features[:num_samples]
    motion_features = motion_features[:num_samples]
    text_features = text_features / np.clip(np.linalg.norm(text_features, axis=1, keepdims=True), 1e-8, None)
    motion_features = motion_features / np.clip(np.linalg.norm(motion_features, axis=1, keepdims=True), 1e-8, None)
    num_groups = max(1, num_samples // group_size)
    hits: list[bool] = []
    for group_idx in range(num_groups):
        start = group_idx * group_size
        end = min(num_samples, start + group_size)
        sims = text_features[start:end] @ motion_features[start:end].T
        targets = np.arange(end - start)
        topk = np.argsort(-sims, axis=1)[:, :k]
        hits.extend([target in topk[row_idx] for row_idx, target in enumerate(targets)])
    return float(np.mean(hits)) if hits else 0.0



def validate_submission_frame(submission_df: pd.DataFrame, expected_rows: int = 3000) -> None:
    required_columns = ["id", "base_tokens", "residual_1", "residual_2", "residual_3", "residual_4", "residual_5"]
    missing = [column for column in required_columns if column not in submission_df.columns]
    if missing:
        raise ValueError(f"Missing submission columns: {missing}")
    if len(submission_df) != expected_rows:
        raise ValueError(f"Expected {expected_rows} rows, found {len(submission_df)}")
    if submission_df[required_columns].isna().any().any():
        raise ValueError("Submission contains missing values")

    for _, row in submission_df.iterrows():
        layer_lengths = []
        for column in TOKEN_COLUMNS:
            tokens = parse_token_string(row[column])
            if not (MIN_LENGTH <= len(tokens) <= MAX_LENGTH):
                raise ValueError(f"Sequence length {len(tokens)} is outside [{MIN_LENGTH}, {MAX_LENGTH}]")
            if any(token < 0 or token >= RVQ_VOCAB_SIZE for token in tokens):
                raise ValueError(f"Token out of range in column {column}")
            layer_lengths.append(len(tokens))
        if len(set(layer_lengths)) != 1:
            raise ValueError("All six RVQ layers must have identical length")


In [9]:
def resolve_data_file(filename: str, env_var: str = "MOTION_S_DATA_ROOT") -> Path:
    configured = os.environ.get(env_var)
    if configured:
        configured_path = Path(configured)
        if configured_path.is_file():
            return configured_path
        if configured_path.is_dir():
            match = find_file_by_name(filename, [configured_path])
            if match is not None:
                return match

    match = find_file_by_name(filename, [Path("/kaggle/input"), ROOT])
    if match is not None:
        return match
    raise FileNotFoundError(f"Could not find {filename}. Set {env_var} or place the competition files under /kaggle/input.")


train_csv = resolve_data_file("train.csv")
test_csv = resolve_data_file("test.csv")
train_df = load_motion_dataframe(train_csv)
test_df = load_motion_dataframe(test_csv)
train_df, val_df = group_aware_train_val_split(train_df, val_fraction=0.10, seed=42)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
display(train_df.head(2))

train: (11220, 11)
val: (1247, 11)
test: (3000, 4)


,id,sentence,gloss,bvh_path,base_tokens,residual_1,residual_2,residual_3,residual_4,residual_5,length
0,72733,"For 381 days, the buses of Montgomery travelle...",BUS MONTGOMERY EMPTY EMPTY EMPTY EMPTY EMPTY E...,dataset/72733/72733.bvh,"[379, 153, 153, 153, 153, 153, 254, 254, 153, ...","[339, 441, 87, 87, 286, 108, 108, 290, 308, 44...","[338, 324, 11, 211, 295, 169, 211, 430, 259, 3...","[406, 429, 457, 457, 457, 457, 457, 457, 457, ...","[64, 357, 137, 456, 353, 305, 95, 95, 498, 498...","[312, 297, 409, 16, 396, 176, 367, 450, 367, 3...",1254
1,5648041,On Friday I go to the market to get some potat...,FRIDAY ME MARKET GO GET POTATO AND EGG//,dataset/5648041/5648041.bvh,"[130, 276, 174, 174, 174, 174, 174, 174, 174, ...","[339, 194, 389, 88, 88, 88, 88, 333, 287, 87, ...","[406, 406, 452, 202, 321, 321, 321, 321, 325, ...","[351, 316, 308, 253, 236, 236, 236, 424, 76, 3...","[347, 125, 500, 95, 274, 424, 424, 498, 499, 4...","[119, 261, 283, 241, 455, 406, 367, 394, 372, ...",172


In [10]:
set_seed(42)

OUTPUT_DIR = ensure_dir(Path("/kaggle/working") / "motion_s_outputs")
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
T5_MODEL_NAME = "t5-small"
LENGTH_BINS = 128
TEXT_MAX_LENGTH = 128
TRAIN_T5 = False
FREEZE_T5_ENCODER = True
FINETUNE_EPOCHS = 4
TRAIN_BATCH_SIZE = 8
INFERENCE_BATCH_SIZE = 4
import torch.nn as nn



def load_local_or_hf_tokenizer(model_name: str, tokenizer_cls, local_files_only: bool = True, **kwargs):
    try:
        return tokenizer_cls.from_pretrained(model_name, local_files_only=local_files_only, **kwargs)
    except Exception:
        return tokenizer_cls.from_pretrained(model_name, **kwargs)



def load_local_or_hf_model(model_name: str, model_cls, local_files_only: bool = True, **kwargs):
    try:
        return model_cls.from_pretrained(model_name, local_files_only=local_files_only, **kwargs)
    except Exception:
        return model_cls.from_pretrained(model_name, **kwargs)



def mean_pool_hidden_states(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(hidden_states)
    summed = (hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp_min(1.0)
    return summed / counts



def encode_clip_texts(texts: Sequence[str], tokenizer: CLIPTokenizer, model: CLIPTextModel, batch_size: int = 64) -> torch.Tensor:
    model = model.to(DEVICE).eval()
    embeddings: list[torch.Tensor] = []
    clip_max_length = min(TEXT_MAX_LENGTH, int(getattr(tokenizer, "model_max_length", TEXT_MAX_LENGTH)), 77)
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc="clip-encode"):
            batch_texts = list(texts[start : start + batch_size])
            tokenized = tokenizer(batch_texts, padding=True, truncation=True, max_length=clip_max_length, return_tensors="pt")
            tokenized = {key: value.to(DEVICE) for key, value in tokenized.items()}
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                outputs = model(**tokenized)
            pooled = mean_pool_hidden_states(outputs.last_hidden_state, tokenized["attention_mask"])
            embeddings.append(pooled.detach().cpu())
    return torch.cat(embeddings, dim=0)



def encode_t5_texts(texts: Sequence[str], tokenizer: T5Tokenizer, model: T5ForConditionalGeneration, batch_size: int = 64) -> torch.Tensor:
    encoder = model.get_encoder() if hasattr(model, "get_encoder") else model.encoder
    encoder = encoder.to(DEVICE).eval()
    embeddings: list[torch.Tensor] = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc="t5-encode"):
            batch_texts = list(texts[start : start + batch_size])
            tokenized = tokenizer(batch_texts, padding=True, truncation=True, max_length=TEXT_MAX_LENGTH, return_tensors="pt")
            tokenized = {key: value.to(DEVICE) for key, value in tokenized.items()}
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                outputs = encoder(input_ids=tokenized["input_ids"], attention_mask=tokenized["attention_mask"], return_dict=True)
            pooled = mean_pool_hidden_states(outputs.last_hidden_state, tokenized["attention_mask"])
            embeddings.append(pooled.detach().cpu())
    return torch.cat(embeddings, dim=0)


class LengthEstimatorHead(nn.Module):
    def __init__(self, input_dim: int = 512, hidden_dim: int = 256, num_bins: int = LENGTH_BINS) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_bins),
        )

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        return self.net(embeddings)



def build_length_bin_centers(num_bins: int = LENGTH_BINS) -> np.ndarray:
    return np.rint(np.linspace(MIN_LENGTH, MAX_LENGTH, num_bins)).astype(np.int64)



def load_length_estimator(checkpoint_path: str | Path, device: torch.device = DEVICE, input_dim: int = 512, num_bins: int = LENGTH_BINS) -> tuple[nn.Module, np.ndarray]:
    payload = torch.load(checkpoint_path, map_location=device)
    if isinstance(payload, nn.Module):
        model = payload.to(device)
    else:
        model = LengthEstimatorHead(input_dim=input_dim, num_bins=num_bins).to(device)
        state_dict = payload.get("model_state_dict", payload.get("state_dict", payload)) if isinstance(payload, dict) else payload
        model.load_state_dict(state_dict, strict=False)
    model.eval()
    return model, build_length_bin_centers(num_bins=num_bins)



def predict_lengths_from_embeddings(
    embeddings: torch.Tensor,
    model: nn.Module | None,
    bin_centers: np.ndarray | None,
    texts: Sequence[str] | None = None,
    fallback_regressor = None,
) -> np.ndarray:
    if model is not None and bin_centers is not None:
        with torch.no_grad():
            logits = model(embeddings.to(DEVICE))
            pred_bins = logits.argmax(dim=-1).detach().cpu().numpy()
        return np.asarray([clip_length(bin_centers[int(bin_idx)]) for bin_idx in pred_bins], dtype=np.int64)
    if fallback_regressor is not None and texts is not None:
        return fallback_regressor.predict(texts)
    raise ValueError("A length model or fallback regressor is required")


vae = None
vae_path = None
try:
    vae_path = resolve_model_file("rvq_vae_best.pth", env_var="MOTION_S_VAE_CKPT")
    vae = torch.load(vae_path, map_location=DEVICE)
    if hasattr(vae, "eval"):
        vae = vae.eval()
except FileNotFoundError:
    print("Warning: rvq_vae_best.pth not found; VAE verification will be skipped.")

clip_tokenizer = load_local_or_hf_tokenizer(CLIP_MODEL_NAME, CLIPTokenizer)
clip_model = load_local_or_hf_model(CLIP_MODEL_NAME, CLIPTextModel).to(DEVICE).eval()

length_model = None
length_bin_centers = None
length_path = None
try:
    length_path = resolve_model_file("length_estimator.pth", env_var="MOTION_S_LENGTH_CKPT")
    length_model, length_bin_centers = load_length_estimator(length_path, device=DEVICE, input_dim=512, num_bins=LENGTH_BINS)
except FileNotFoundError:
    print("Warning: length_estimator.pth not found; using gloss-length fallback.")

t5_tokenizer = load_local_or_hf_tokenizer(T5_MODEL_NAME, T5Tokenizer)
t5_tokenizer.add_special_tokens({"additional_special_tokens": LAYER_TOKENS + RVQ_TOKENS})
t5_model = load_local_or_hf_model(T5_MODEL_NAME, T5ForConditionalGeneration).to(DEVICE)
t5_model.resize_token_embeddings(len(t5_tokenizer))
t5_model.config.use_cache = False
t5_model.config.decoder_start_token_id = t5_tokenizer.pad_token_id
t5_model = t5_model.eval()

train_gloss_texts = [build_gloss_prompt(gloss) for gloss in train_df["gloss"]]
val_gloss_texts = [build_gloss_prompt(gloss) for gloss in val_df["gloss"]]
test_gloss_texts = [build_gloss_prompt(gloss) for gloss in test_df["gloss"]]
fallback_length_regressor = GlossLengthRegressor().fit(train_gloss_texts, train_df["length"]) if length_model is None else None

train_clip_embeddings = encode_clip_texts(train_gloss_texts, clip_tokenizer, clip_model)
val_clip_embeddings = encode_clip_texts(val_gloss_texts, clip_tokenizer, clip_model)
test_clip_embeddings = encode_clip_texts(test_gloss_texts, clip_tokenizer, clip_model)

print("VAE:", type(vae).__name__)
print("Length model:", type(length_model).__name__)
print("T5 tokenizer size:", len(t5_tokenizer))
print("CLIP embeddings:", train_clip_embeddings.shape, val_clip_embeddings.shape, test_clip_embeddings.shape)

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 32497.28it/s]
CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
visual_projection.weight                                       | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_

clip-encode: 100%|██████████| 47/47 [02:40<00:00,  3.42s/it]

VAE: NoneType
Length model: NoneType
T5 tokenizer size: 622
CLIP embeddings: torch.Size([11220, 512]) torch.Size([1247, 512]) torch.Size([3000, 512])


## Data Loading and Preprocessing

The competition CSVs already contain the text inputs plus the 6 RVQ token layers.

Design choice: split by `signer_id` when possible so the validation set is less likely to leak signer-specific motion style.
That gives a cleaner estimate of retrieval quality than a purely random split.

In [ ]:
train_pred_lengths = predict_lengths_from_embeddings(train_clip_embeddings, length_model, length_bin_centers, train_gloss_texts, fallback_length_regressor)
val_pred_lengths = predict_lengths_from_embeddings(val_clip_embeddings, length_model, length_bin_centers, val_gloss_texts, fallback_length_regressor)
test_pred_lengths = predict_lengths_from_embeddings(test_clip_embeddings, length_model, length_bin_centers, test_gloss_texts, fallback_length_regressor)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df["pred_length"] = train_pred_lengths
val_df["pred_length"] = val_pred_lengths
test_df["pred_length"] = test_pred_lengths

train_df["true_length"] = train_df["length"].apply(lambda value: clip_length(value) if pd.notna(value) else np.nan)
val_df["true_length"] = val_df["length"].apply(lambda value: clip_length(value) if pd.notna(value) else np.nan)

print("train pred length range:", int(train_df["pred_length"].min()), int(train_df["pred_length"].max()))
print("val pred length range:", int(val_df["pred_length"].min()), int(val_df["pred_length"].max()))
print("test pred length range:", int(test_df["pred_length"].min()), int(test_df["pred_length"].max()))
print("train/val true lengths:", int(train_df["true_length"].min()), int(train_df["true_length"].max()), int(val_df["true_length"].min()), int(val_df["true_length"].max()))
display(test_df[["id", "sentence", "gloss", "pred_length"]].head(3))

train pred length range: 47 800
val pred length range: 78 800
test pred length range: 47 800
train/val true lengths: 40 800 40 740


,id,sentence,gloss,pred_length
0,6420249,Mary never told me she was a vegetarian.,ME NEVER TOLD ME SHE VEGETARIAN//,126
1,6420682,Mary told me that she's doing that now.,NOW SHE DO THAT//,94
2,6425789,Mary told me that she was tense.,TENSE SHE TOLD ME//,94


## Model Loading and Length Prediction

This stage loads the three model components and turns gloss text into a predicted motion length:

- CLIP text encoding for the length estimator input
- `length_estimator.pth` for the discrete length bin prediction
- `t5-small` with added RVQ/layer tokens for sequence generation

The predicted length is clipped to the competition range before generation starts.

In [12]:
MAX_TARGET_LENGTH = NUM_RVQ_LAYERS * (MAX_LENGTH + 1)


class GlossRVQDataset(Dataset):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame.reset_index(drop=True).copy()
        self.texts = [build_gloss_prompt(gloss) for gloss in self.frame["gloss"]]
        self.targets = [encode_rvq_layers([row[column] for column in TOKEN_COLUMNS]) for _, row in self.frame.iterrows()]

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, str]:
        return {"text": self.texts[index], "target": self.targets[index]}



def collate_t5_batch(batch: list[dict[str, str]]) -> dict[str, torch.Tensor]:
    input_texts = [item["text"] for item in batch]
    target_texts = [item["target"] for item in batch]
    input_tokens = t5_tokenizer(
        input_texts,
        padding=True,
        truncation=True,
        max_length=TEXT_MAX_LENGTH,
        return_tensors="pt",
    )
    target_tokens = t5_tokenizer(
        text_target=target_texts,
        padding=True,
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors="pt",
    )
    labels = target_tokens["input_ids"].clone()
    labels[labels == t5_tokenizer.pad_token_id] = -100
    return {
        "input_ids": input_tokens["input_ids"],
        "attention_mask": input_tokens["attention_mask"],
        "labels": labels,
        "target_texts": target_texts,
        "input_texts": input_texts,
    }


train_dataset = GlossRVQDataset(train_df)
val_dataset = GlossRVQDataset(val_df)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_t5_batch)
val_loader = DataLoader(val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False, collate_fn=collate_t5_batch)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("example target:")
print(train_dataset.targets[0][:400] + "...")

train batches: 1403
val batches: 156
example target:
<layer_0> <rvq_379> <rvq_153> <rvq_153> <rvq_153> <rvq_153> <rvq_153> <rvq_254> <rvq_254> <rvq_153> <rvq_153> <rvq_153> <rvq_201> <rvq_306> <rvq_214> <rvq_134> <rvq_134> <rvq_134> <rvq_134> <rvq_134> <rvq_134> <rvq_134> <rvq_352> <rvq_30> <rvq_30> <rvq_236> <rvq_372> <rvq_360> <rvq_360> <rvq_360> <rvq_312> <rvq_312> <rvq_312> <rvq_312> <rvq_312> <rvq_312> <rvq_312> <rvq_312> <rvq_312> <rvq_312> <r...


## T5 Dataset and Targets

The dataset keeps gloss text as the input side and flattens the six RVQ layers into a single token string target:

- `layer_0` through `layer_5` markers separate the layers
- `<rvq_0>` through `<rvq_511>` represent the RVQ codes
- Rows are trimmed to a shared length when the source layers are not perfectly aligned

This format is used for optional T5 fine-tuning and for validation of generated outputs.

In [13]:
sample_batch = next(iter(train_loader))
print("input_ids:", sample_batch["input_ids"].shape)
print("attention_mask:", sample_batch["attention_mask"].shape)
print("labels:", sample_batch["labels"].shape)
print("first input:", sample_batch["input_texts"][0])
print("first target prefix:", sample_batch["target_texts"][0][:200] + "...")

input_ids: torch.Size([8, 15])
attention_mask: torch.Size([8, 15])
labels: torch.Size([8, 829])
first input: SHOP POLICE SUPERVISION KEEP//
first target prefix: <layer_0> <rvq_379> <rvq_295> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_71> <rvq_205> <rvq_221> <rvq_262> <rvq_262> <rvq_262> <rvq_262> <rvq_262> <...


## Optional T5 Fine-Tuning

If time and memory allow, the notebook fine-tunes `t5-small` on gloss -> flattened RVQ token strings.

Defaults:
- AdamW optimizer
- learning rate `3e-4`
- 3 to 5 epochs
- batch size `8`
- gradient checkpointing enabled for the seq2seq model

Leave `TRAIN_T5 = False` to skip training and rely on the pretrained model plus fallback retrieval.

In [14]:
T5_CHECKPOINT = OUTPUT_DIR / "t5_small_motion_s_best.pt"


if TRAIN_T5:
    if hasattr(t5_model, "gradient_checkpointing_enable"):
        t5_model.gradient_checkpointing_enable()
    t5_model.config.use_cache = False
    if hasattr(t5_model, "encoder") and FREEZE_T5_ENCODER:
        for parameter in t5_model.encoder.parameters():
            parameter.requires_grad = False

    optimizer = torch.optim.AdamW((parameter for parameter in t5_model.parameters() if parameter.requires_grad), lr=3e-4)
    total_steps = max(1, FINETUNE_EPOCHS * len(train_loader))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(0.1 * total_steps)),
        num_training_steps=total_steps,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    best_val_loss = float("inf")
    history: list[dict[str, float]] = []

    for epoch in range(FINETUNE_EPOCHS):
        t5_model.train()
        running_train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"train-epoch-{epoch + 1}"):
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                outputs = t5_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(t5_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_train_loss += float(loss.item())

        t5_model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"val-epoch-{epoch + 1}"):
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
                outputs = t5_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                running_val_loss += float(outputs.loss.item())

        train_loss = running_train_loss / max(1, len(train_loader))
        val_loss = running_val_loss / max(1, len(val_loader))
        metrics = {"epoch": float(epoch + 1), "train_loss": train_loss, "val_loss": val_loss}
        history.append(metrics)
        print(metrics)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({"model_state_dict": t5_model.state_dict(), "tokenizer_size": len(t5_tokenizer)}, T5_CHECKPOINT)

    history_df = pd.DataFrame(history)
    display(history_df)
else:
    print("T5 fine-tuning skipped. Set TRAIN_T5 = True to train the seq2seq token generator.")

T5 fine-tuning skipped. Set TRAIN_T5 = True to train the seq2seq token generator.


## Inference and Submission Generation

For each test sample:
1. Encode the gloss with T5
2. Predict the motion length with the CLIP-based estimator
3. Autoregressively generate the flattened RVQ token sequence
4. Split the sequence back into six equal layers
5. Fall back to nearest-neighbor retrieval or random tokens if validation fails
6. Save the final `submission.csv`

All outputs are validated before the CSV is written.

In [ ]:
def tokens_to_string(tokens: Sequence[int]) -> str:
    return " ".join(str(int(token)) for token in tokens)



def greedy_decode_fixed_length(model: T5ForConditionalGeneration, input_ids: torch.Tensor, attention_mask: torch.Tensor, total_steps: int) -> torch.Tensor:
    model = model.to(DEVICE).eval()
    batch_size = input_ids.size(0)
    decoder_input_ids = torch.full(
        (batch_size, 1),
        fill_value=t5_tokenizer.pad_token_id,
        dtype=torch.long,
        device=DEVICE,
    )
    generated_steps: list[torch.Tensor] = []
    with torch.no_grad():
        for _ in range(total_steps):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                use_cache=False,
                return_dict=True,
            )
            next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            decoder_input_ids = torch.cat([decoder_input_ids, next_token], dim=1)
            generated_steps.append(next_token)
    return torch.cat(generated_steps, dim=1)



def decode_rvq_layers_from_tokens(token_ids: Sequence[int], expected_length: int) -> list[list[int]] | None:
    tokens = t5_tokenizer.convert_ids_to_tokens(list(token_ids), skip_special_tokens=False)
    layers = [[] for _ in range(NUM_RVQ_LAYERS)]
    current_layer = None
    flat_codes: list[int] = []
    for token in tokens:
        if token in LAYER_TOKENS:
            current_layer = LAYER_TOKENS.index(token)
            continue
        if token.startswith("<rvq_") and token.endswith(">"):
            try:
                value = int(token[5:-1])
            except ValueError:
                continue
            value = int(np.clip(value, 0, RVQ_VOCAB_SIZE - 1))
            flat_codes.append(value)
            if current_layer is not None:
                layers[current_layer].append(value)

    if all(len(layer) == expected_length for layer in layers):
        return layers
    if len(flat_codes) >= NUM_RVQ_LAYERS * expected_length:
        return [flat_codes[idx * expected_length : (idx + 1) * expected_length] for idx in range(NUM_RVQ_LAYERS)]
    return None



def nearest_neighbor_fallback(sample_text: str, expected_length: int, sample_embedding: torch.Tensor) -> list[list[int]]:
    del sample_text
    train_norm = train_clip_embeddings / torch.clamp(train_clip_embeddings.norm(dim=1, keepdim=True), min=1e-8)
    sample_norm = sample_embedding / torch.clamp(sample_embedding.norm(dim=0, keepdim=True), min=1e-8)
    best_index = int(torch.argmax(train_norm @ sample_norm).item())
    best_row = train_df.iloc[best_index]
    fallback_layers = normalize_layer_tokens([best_row[column] for column in TOKEN_COLUMNS])
    adjusted_layers: list[list[int]] = []
    rng = np.random.default_rng(42 + expected_length)
    for layer in fallback_layers:
        if len(layer) >= expected_length:
            adjusted_layers.append([int(token) for token in layer[:expected_length]])
        else:
            padded = [int(token) for token in layer]
            while len(padded) < expected_length:
                padded.append(int(rng.integers(0, RVQ_VOCAB_SIZE)))
            adjusted_layers.append(padded[:expected_length])
    return adjusted_layers



generation_model = t5_model
if T5_CHECKPOINT.exists():
    payload = torch.load(T5_CHECKPOINT, map_location=DEVICE)
    state_dict = payload.get("model_state_dict", payload) if isinstance(payload, dict) else payload
    generation_model.load_state_dict(state_dict, strict=False)
    generation_model = generation_model.to(DEVICE).eval()
    print("Loaded fine-tuned T5 checkpoint:", T5_CHECKPOINT)
else:
    print("No fine-tuned checkpoint found. Using the current T5 weights and fallback retrieval when needed.")

submission_rows: list[dict[str, object]] = []
validated_bundles: list[list[list[int]]] = []

for start in tqdm(range(0, len(test_df), INFERENCE_BATCH_SIZE), desc="generate-test"):
    end = min(len(test_df), start + INFERENCE_BATCH_SIZE)
    batch_frame = test_df.iloc[start:end].reset_index(drop=True)
    batch_texts = [build_gloss_prompt(gloss) for gloss in batch_frame["gloss"]]
    batch_lengths = [int(length) for length in batch_frame["pred_length"]]
    batch_tokens = t5_tokenizer(batch_texts, padding=True, truncation=True, max_length=TEXT_MAX_LENGTH, return_tensors="pt")
    input_ids = batch_tokens["input_ids"].to(DEVICE)
    attention_mask = batch_tokens["attention_mask"].to(DEVICE)
    total_steps = NUM_RVQ_LAYERS * (max(batch_lengths) + 1)
    generated_ids = greedy_decode_fixed_length(generation_model, input_ids, attention_mask, total_steps=total_steps)

    for row_idx, row in batch_frame.iterrows():
        expected_length = int(row["pred_length"])
        expected_steps = NUM_RVQ_LAYERS * (expected_length + 1)
        candidate = decode_rvq_layers_from_tokens(generated_ids[row_idx, :expected_steps].detach().cpu().tolist(), expected_length)
        if candidate is None:
            candidate = nearest_neighbor_fallback(batch_texts[row_idx], expected_length, test_clip_embeddings[start + row_idx])
        candidate = normalize_layer_tokens(candidate)
        if len(candidate) != NUM_RVQ_LAYERS or any(len(layer) != expected_length for layer in candidate):
            rng = np.random.default_rng(1234 + int(row["id"]))
            candidate = [[int(rng.integers(0, RVQ_VOCAB_SIZE)) for _ in range(expected_length)] for _ in range(NUM_RVQ_LAYERS)]

        validated_bundles.append(candidate)
        submission_rows.append(
            {
                "id": row["id"],
                "base_tokens": tokens_to_string(candidate[0]),
                "residual_1": tokens_to_string(candidate[1]),
                "residual_2": tokens_to_string(candidate[2]),
                "residual_3": tokens_to_string(candidate[3]),
                "residual_4": tokens_to_string(candidate[4]),
                "residual_5": tokens_to_string(candidate[5]),
            }
        )

submission_df = pd.DataFrame(submission_rows)
validate_submission_frame(submission_df)
submission_path = OUTPUT_DIR / "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(submission_path)
display(submission_df.head(2))

No fine-tuned checkpoint found. Using the current T5 weights and fallback retrieval when needed.


generate-test:   0%|          | 0/750 [00:00<?, ?it/s]

## Local Evaluation

This slice checks the notebook against the competition-style proxies:

- token histogram FID on the RVQ bundles
- T5-space R-Precision proxy between gloss text and generated token strings
- diversity from pairwise feature distance
- optional VAE verification when the checkpoint exposes a decode method

The goal is to catch invalid token bundles before submission.

In [ ]:
def bundles_to_tensor(bundles: list[list[list[int]]]) -> tuple[torch.Tensor, torch.Tensor]:
    lengths = [len(bundle[0]) for bundle in bundles]
    max_length = max(lengths)
    padded_bundles = []
    masks = []
    for bundle, length in zip(bundles, lengths):
        tensor = torch.tensor(bundle, dtype=torch.long)
        if length < max_length:
            pad = torch.full((NUM_RVQ_LAYERS, max_length - length), -100, dtype=torch.long)
            tensor = torch.cat([tensor, pad], dim=-1)
        padded_bundles.append(tensor)
        masks.append(torch.arange(max_length) < length)
    return torch.stack(padded_bundles, dim=0), torch.stack(masks, dim=0)



def frame_to_real_bundle(frame: pd.DataFrame) -> tuple[torch.Tensor, torch.Tensor]:
    bundles = []
    for _, row in frame.iterrows():
        layers = normalize_layer_tokens([row[column] for column in TOKEN_COLUMNS])
        bundles.append(layers)
    return bundles_to_tensor(bundles)



def try_vae_decode(bundle: list[list[int]]):
    if not hasattr(vae, "decode") and not hasattr(vae, "decode_tokens"):
        return None
    candidate = torch.tensor(bundle, dtype=torch.long, device=DEVICE).unsqueeze(0)
    for method_name in ("decode_tokens", "decode"):
        method = getattr(vae, method_name, None)
        if callable(method):
            try:
                return method(candidate)
            except Exception:
                continue
    return None



eval_size = min(64, len(val_df))
eval_indices = np.random.default_rng(42).choice(len(val_df), size=eval_size, replace=False)
eval_df = val_df.iloc[eval_indices].reset_index(drop=True)
eval_clip_subset = val_clip_embeddings[eval_indices]
real_bundle, real_mask = frame_to_real_bundle(eval_df)
eval_text_features = encode_t5_texts([build_gloss_prompt(gloss) for gloss in eval_df["gloss"]], t5_tokenizer, generation_model)

eval_pred_bundles: list[list[list[int]]] = []
for start in tqdm(range(0, len(eval_df), INFERENCE_BATCH_SIZE), desc="eval-generate"):
    end = min(len(eval_df), start + INFERENCE_BATCH_SIZE)
    batch_frame = eval_df.iloc[start:end].reset_index(drop=True)
    batch_texts = [build_gloss_prompt(gloss) for gloss in batch_frame["gloss"]]
    batch_lengths = [int(length) for length in batch_frame["pred_length"]]
    batch_tokens = t5_tokenizer(batch_texts, padding=True, truncation=True, max_length=TEXT_MAX_LENGTH, return_tensors="pt")
    input_ids = batch_tokens["input_ids"].to(DEVICE)
    attention_mask = batch_tokens["attention_mask"].to(DEVICE)
    total_steps = NUM_RVQ_LAYERS * (max(batch_lengths) + 1)
    generated_ids = greedy_decode_fixed_length(generation_model, input_ids, attention_mask, total_steps=total_steps)

    for row_idx, row in batch_frame.iterrows():
        expected_length = int(row["pred_length"])
        expected_steps = NUM_RVQ_LAYERS * (expected_length + 1)
        candidate = decode_rvq_layers_from_tokens(generated_ids[row_idx, :expected_steps].detach().cpu().tolist(), expected_length)
        if candidate is None:
            candidate = nearest_neighbor_fallback(batch_texts[row_idx], expected_length, eval_clip_subset[start + row_idx])
        candidate = normalize_layer_tokens(candidate)
        if len(candidate) != NUM_RVQ_LAYERS or any(len(layer) != expected_length for layer in candidate):
            rng = np.random.default_rng(1234 + int(row["id"]))
            candidate = [[int(rng.integers(0, RVQ_VOCAB_SIZE)) for _ in range(expected_length)] for _ in range(NUM_RVQ_LAYERS)]
        eval_pred_bundles.append(candidate)

pred_bundle, pred_mask = bundles_to_tensor(eval_pred_bundles)
real_features = token_histogram_features(real_bundle, real_mask)
pred_features = token_histogram_features(pred_bundle, pred_mask)
real_text_embeddings = eval_text_features.numpy()
pred_text_embeddings = encode_t5_texts([encode_rvq_layers(bundle) for bundle in eval_pred_bundles], t5_tokenizer, generation_model).numpy()

fid_proxy = approximate_fid(real_features, pred_features)
r_precision_proxy = r_precision_at_k(real_text_embeddings, pred_text_embeddings, k=3, group_size=min(32, eval_size))
diversity_proxy = diversity_score(pred_features)
vae_preview = try_vae_decode(eval_pred_bundles[0])

print({
    "fid_proxy": fid_proxy,
    "r_precision_proxy": r_precision_proxy,
    "diversity_proxy": diversity_proxy,
    "vae_preview_available": vae_preview is not None,
})

generate: 100%|██████████| 2/2 [00:00<00:00,  3.59it/s]


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3073 is different from 256)

## Fallback Strategy

The fallback path is deterministic and submission-safe:

- try the T5 decoder first
- if the generated bundle fails validation, copy the nearest training example by CLIP similarity
- if that still fails, sample random RVQ tokens in the legal range

The fallback only activates for rows that do not satisfy the length or range checks.

In [ ]:
def random_bundle(expected_length: int, seed: int = 42) -> list[list[int]]:
    rng = np.random.default_rng(seed)
    return [[int(rng.integers(0, RVQ_VOCAB_SIZE)) for _ in range(expected_length)] for _ in range(NUM_RVQ_LAYERS)]


print("Fallback strategies ready: nearest-neighbor copy first, random token bundles as the last resort.")

## How to Improve Further

Good next steps after the baseline:

- unfreeze the top T5 layers after the decoder stabilizes
- add a length token or length bin embedding to the T5 prompt
- try layer-wise training if the flattened target is too long for your GPU
- use signer-aware conditioning when the split allows it
- increase diversity with top-k or nucleus sampling during generation
- cache CLIP embeddings so nearest-neighbor fallback stays fast

The notebook is intentionally structured so these changes can be added without rewriting the submission path.